# Visiting Card Reader
Extract **Name**, **Email**, **Phone**, and **Address** from business card images using Mistral (Azure AI) with a Streamlit UI embedded in this notebook.

## Step 1 — Install dependencies

In [ ]:
%pip install -q streamlit azure-ai-inference Pillow python-dotenv stlite-ipython

## Step 2 — Core extraction logic

In [1]:
import base64
import io
import json
import os
from dotenv import load_dotenv
from PIL import Image, ImageChops
from azure.ai.inference import ChatCompletionsClient
from azure.core.credentials import AzureKeyCredential

load_dotenv()

ENDPOINT = "https://mistral-small-2503-car-damage.swedencentral.models.ai.azure.com"
API_VERSION = "2024-05-01-preview"

SYSTEM_PROMPT = """You are an expert at reading business / visiting cards.
Extract the following fields from the card image and return ONLY valid JSON — no markdown, no explanation.

Required JSON schema:
{
  "name": "full name or null",
  "email": "email address or null",
  "phone": "phone number(s) as a single string or null",
  "address": "full address as a single string or null"
}

If a field is not visible on the card set it to null."""


def image_to_base64(image: Image.Image, quality: int = 95) -> str:
    """Convert a PIL Image to a base64-encoded JPEG data-URI."""
    if image.mode != "RGB":
        image = image.convert("RGB")
    buf = io.BytesIO()
    image.save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    encoded = base64.b64encode(buf.getvalue()).decode("utf-8")
    return f"data:image/jpeg;base64,{encoded}"


def crop_whitespace(image: Image.Image) -> Image.Image:
    """Trim uniform white borders from a card scan."""
    bg = Image.new("RGB", image.size, (255, 255, 255))
    diff = ImageChops.difference(image.convert("RGB"), bg)
    bbox = diff.getbbox()
    return image.crop(bbox) if bbox else image


def extract_card_info(image: Image.Image) -> dict:
    """Send the card image to Mistral and return parsed JSON fields."""
    api_key = os.getenv("AZURE_INFERENCE_CREDENTIAL", "")
    if not api_key:
        raise ValueError("AZURE_INFERENCE_CREDENTIAL env variable is not set.")

    client = ChatCompletionsClient(
        endpoint=ENDPOINT,
        credential=AzureKeyCredential(api_key),
        api_version=API_VERSION,
    )

    prepped = crop_whitespace(image)
    img_uri = image_to_base64(prepped)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {"url": img_uri},
                },
                {
                    "type": "text",
                    "text": "Extract the contact information from this visiting card.",
                },
            ],
        },
    ]

    response = client.complete({"messages": messages, "max_tokens": 512, "temperature": 0.0})
    raw = response.choices[0].message.content.strip()

    # Strip markdown fences if the model adds them
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    raw = raw.strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"name": None, "email": None, "phone": None, "address": None, "raw_response": raw}


print("✓ Extraction logic loaded.")

✓ Extraction logic loaded.


## Step 3 — Write the Streamlit app to a `.py` file

In [2]:
streamlit_code = '''
import base64, io, json, os, sys
from pathlib import Path
import streamlit as st
from PIL import Image, ImageChops
from azure.ai.inference import ChatCompletionsClient
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv()

ENDPOINT    = "https://mistral-small-2503-car-damage.swedencentral.models.ai.azure.com"
API_VERSION = "2024-05-01-preview"

SYSTEM_PROMPT = """You are an expert at reading business / visiting cards.
Extract the following fields from the card image and return ONLY valid JSON — no markdown, no explanation.

Required JSON schema:
{
  "name": "full name or null",
  "email": "email address or null",
  "phone": "phone number(s) as a single string or null",
  "address": "full address as a single string or null"
}

If a field is not visible on the card set it to null."""


def _image_to_b64(img: Image.Image) -> str:
    if img.mode != "RGB":
        img = img.convert("RGB")
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=95)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()


def _crop_whitespace(img: Image.Image) -> Image.Image:
    bg   = Image.new("RGB", img.size, (255, 255, 255))
    diff = ImageChops.difference(img.convert("RGB"), bg)
    bbox = diff.getbbox()
    return img.crop(bbox) if bbox else img


def extract_card_info(image: Image.Image) -> dict:
    api_key = os.getenv("AZURE_INFERENCE_CREDENTIAL", "")
    if not api_key:
        st.error("AZURE_INFERENCE_CREDENTIAL is not set in your .env file.")
        return {}
    client = ChatCompletionsClient(
        endpoint=ENDPOINT,
        credential=AzureKeyCredential(api_key),
        api_version=API_VERSION,
    )
    prepped = _crop_whitespace(image)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": _image_to_b64(prepped)}},
                {"type": "text",      "text": "Extract the contact information from this visiting card."},
            ],
        },
    ]
    resp = client.complete({"messages": messages, "max_tokens": 512, "temperature": 0.0})
    raw  = resp.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    raw = raw.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"name": None, "email": None, "phone": None, "address": None, "raw_response": raw}


# ── UI ──────────────────────────────────────────────────────────────────────

st.set_page_config(page_title="Visiting Card Reader", layout="centered")
st.title("📇 Visiting Card Reader")
st.caption("Upload a photo or scan of a business card to extract contact details.")

uploaded = st.file_uploader(
    "Upload visiting card image",
    type=["png", "jpg", "jpeg", "webp", "bmp"],
    help="Supports PNG, JPG, JPEG, WEBP, BMP",
)

if uploaded:
    image = Image.open(uploaded)

    col_img, col_res = st.columns([1, 1])

    with col_img:
        st.subheader("Card Preview")
        st.image(image, use_container_width=True)

    with col_res:
        st.subheader("Extracted Information")
        with st.spinner("Analysing card with Mistral..."):
            info = extract_card_info(image)

        if info:
            fields = [
                ("👤 Name",    info.get("name")),
                ("📧 Email",   info.get("email")),
                ("📞 Phone",   info.get("phone")),
                ("🏠 Address", info.get("address")),
            ]

            all_null = all(v is None for _, v in fields)
            if all_null:
                st.warning("No contact information could be extracted from this image.")
            else:
                for label, value in fields:
                    if value:
                        st.markdown(f"**{label}**")
                        st.write(value)
                        st.divider()

            with st.expander("Raw JSON response"):
                st.json(info)

            json_str = json.dumps(info, indent=2)
            st.download_button(
                label="⬇️ Download as JSON",
                data=json_str,
                file_name="visiting_card_info.json",
                mime="application/json",
            )
'''

# Write the app to disk so we can launch it with streamlit
with open("visiting_card_app.py", "w") as f:
    f.write(streamlit_code)

print("✓ visiting_card_app.py written.")

✓ visiting_card_app.py written.


## Step 4 — Launch the Streamlit app (inline in notebook)

The cell below starts Streamlit in a background thread and renders it inside an `IFrame` so you never leave the notebook.

In [3]:
import subprocess
import sys
import time
import threading
from IPython.display import IFrame, display

PORT = 8502

def _run_streamlit():
    subprocess.run(
        [sys.executable, "-m", "streamlit", "run", "visiting_card_app.py",
         "--server.port", str(PORT),
         "--server.headless", "true",
         "--browser.gatherUsageStats", "false"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

thread = threading.Thread(target=_run_streamlit, daemon=True)
thread.start()

# Give Streamlit a moment to start up
time.sleep(4)

print(f"Streamlit running on http://localhost:{PORT}")
display(IFrame(src=f"http://localhost:{PORT}", width="100%", height=700))

Streamlit running on http://localhost:8502


## Step 5 — Use the extraction functions directly in Python (no UI needed)

You can also call `extract_card_info` directly from any notebook cell.

In [ ]:
from PIL import Image

# ── Replace with the path to your card image ─────────────────────────────────
card_image_path = r"C:\path\to\your\visiting_card.jpg"   # <-- change this
# ─────────────────────────────────────────────────────────────────────────────

card = Image.open(card_image_path)
info = extract_card_info(card)

print("Name   :", info.get("name"))
print("Email  :", info.get("email"))
print("Phone  :", info.get("phone"))
print("Address:", info.get("address"))